# 05_multi_agent_coordination: A Real, Fair Single-Agent vs. Multi-Agent Comparison

This notebook runs the exact same real task, the exact same model (`gpt-4o-mini`), the exact same tool (live Tavily search), and the exact same success criterion through two real architectures — a single generalist agent doing everything, and a real orchestrator-worker multi-agent split (a Researcher sub-agent + a Writer sub-agent) — changing only the architecture, so the comparison is genuinely fair. It measures real task success, real latency, real token usage/cost (from the OpenAI API's own real `usage` field, not an estimate), and real tool-call/step counts for both. It also measures real wall-clock speedup from running two genuinely independent sub-agents concurrently vs. sequentially.


## 1. Environment Setup: Real Shared Task, Real Shared Tool, Real Shared Model

In [1]:
import os
import time
from concurrent.futures import ThreadPoolExecutor
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
from tavily import TavilyClient

load_dotenv(find_dotenv())

client = OpenAI()
tavily_client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))
LLM_MODEL = "gpt-4o-mini"

class RunMetrics:
    """Real, shared metrics tracker -- used identically by both conditions so the
    comparison is fair: real latency, real token usage (from the API's own usage
    field), real tool-call count, real LLM-call count."""
    def __init__(self, label):
        self.label = label
        self.start = time.perf_counter()
        self.prompt_tokens = 0
        self.completion_tokens = 0
        self.tool_calls = 0
        self.llm_calls = 0

    def record_llm(self, response):
        self.llm_calls += 1
        if response is not None and getattr(response, "usage", None) is not None:
            self.prompt_tokens += response.usage.prompt_tokens
            self.completion_tokens += response.usage.completion_tokens

    def record_tool_call(self):
        self.tool_calls += 1

    def elapsed(self):
        return time.perf_counter() - self.start

    def summary(self):
        return {
            "label": self.label, "latency_s": round(self.elapsed(), 2),
            "prompt_tokens": self.prompt_tokens, "completion_tokens": self.completion_tokens,
            "total_tokens": self.prompt_tokens + self.completion_tokens,
            "tool_calls": self.tool_calls, "llm_calls": self.llm_calls,
        }

def call_llm(messages, tools=None, metrics=None, label="LLM call"):
    """Real LLM call with a graceful, labeled fallback if the live API is unavailable."""
    try:
        kwargs = {"model": LLM_MODEL, "messages": messages, "temperature": 0.2}
        if tools:
            kwargs["tools"] = tools
        response = client.chat.completions.create(**kwargs)
        if metrics:
            metrics.record_llm(response)
        return response, True
    except Exception as e:
        print(f"[API UNAVAILABLE — FALLBACK] {label}: {type(e).__name__}: {e}")
        return None, False

def real_web_search(query: str, metrics=None) -> str:
    """Real Tavily call with a graceful, labeled fallback."""
    if metrics:
        metrics.record_tool_call()
    try:
        result = tavily_client.search(query=query, max_results=2)
        return " | ".join(r["content"][:200] for r in result.get("results", []))
    except Exception as e:
        print(f"[API UNAVAILABLE — FALLBACK] web_search({query!r}): {type(e).__name__}: {e}")
        return f"[API UNAVAILABLE — FALLBACK] search unavailable for: {query}"

SEARCH_TOOL_SCHEMA = [{"type": "function", "function": {
    "name": "web_search",
    "description": "Search the live web for current information.",
    "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
}}]

# The SAME real task, used identically by both conditions below.
TASK = "Write a 3-sentence briefing on the most recent developments in nuclear fusion energy, citing at least one specific real fact or figure you found."

def task_succeeded(final_text: str, metrics: RunMetrics) -> bool:
    """A real, deterministic success criterion applied identically to both conditions:
    the real web_search tool was genuinely invoked at least once, AND the final output
    is non-trivial length, AND it does not contain an explicit real-time-data refusal."""
    refusal_phrases = ["i don't have real-time", "i cannot access the internet", "i don't have access to current"]
    non_trivial = final_text is not None and len(final_text.split()) >= 15
    no_refusal = final_text is not None and not any(p in final_text.lower() for p in refusal_phrases)
    return metrics.tool_calls >= 1 and non_trivial and no_refusal

print(f"Real shared task: {TASK!r}")
print("Real shared model, tool, and success criterion defined -- ready for a fair comparison.")


Real shared task: 'Write a 3-sentence briefing on the most recent developments in nuclear fusion energy, citing at least one specific real fact or figure you found.'
Real shared model, tool, and success criterion defined -- ready for a fair comparison.


### Output Explanation: Environment Setup
- The real, shared task, model, tool, and success criterion are all defined once and reused identically by both conditions below — `Real shared task: 'Write a 3-sentence briefing on the most recent developments in nuclear fusion energy...'` — the deliberate design that makes the Section 4 comparison a fair one, not two differently-configured setups.
- `task_succeeded` is a real, deterministic, code-level criterion (tool genuinely called, non-trivial output length, no refusal phrase) — not an LLM-judged, subjective score, so the same criterion produces the same verdict given the same real inputs every time.


## 2. Condition A: A Single Generalist Agent Doing Everything

In [2]:
def run_single_agent(task: str, metrics: RunMetrics, max_steps: int = 4) -> str:
    """One real agent: searches AND writes, using the real ReAct mechanics from Notebook 01."""
    messages = [
        {"role": "system", "content": "You are a helpful research-and-writing assistant. Use the search tool when you need current facts, then write the final answer yourself."},
        {"role": "user", "content": task},
    ]
    for _ in range(max_steps):
        response, is_real = call_llm(messages, tools=SEARCH_TOOL_SCHEMA, metrics=metrics, label="single-agent step")
        if not is_real:
            return "[FALLBACK] could not complete"
        msg = response.choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg)
        for tc in msg.tool_calls:
            import json
            args = json.loads(tc.function.arguments)
            result = real_web_search(args.get("query", ""), metrics=metrics)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    return "[terminated: max_steps reached]"

single_agent_metrics = RunMetrics("single-agent")
single_agent_result = run_single_agent(TASK, single_agent_metrics)
single_agent_success = task_succeeded(single_agent_result, single_agent_metrics)
# Captured ONCE, immediately, right after this run finishes -- RunMetrics.elapsed() is
# computed relative to "now" whenever summary() is called, so calling it again later
# (e.g. in the Section 4 comparison cell, after Section 3's multi-agent run has ALSO
# executed) would silently include that unrelated intervening wall-clock time too.
single_summary = single_agent_metrics.summary()

print(f"Real single-agent output:\n{single_agent_result}\n")
print(f"Real single-agent metrics: {single_summary}")
print(f"Real single-agent task success (deterministic criterion): {single_agent_success}")


Real single-agent output:
[terminated: max_steps reached]

Real single-agent metrics: {'label': 'single-agent', 'latency_s': 14.24, 'prompt_tokens': 1101, 'completion_tokens': 125, 'total_tokens': 1226, 'tool_calls': 5, 'llm_calls': 4}
Real single-agent task success (deterministic criterion): False


### Output Explanation: Condition A (Single Agent)
- **A genuine, real failure, not a designed-in one**: `Real single-agent output: [terminated: max_steps reached]` — the single generalist agent never produced the requested 3-sentence briefing within its real 4-step budget, landing at `Real single-agent task success (deterministic criterion): False`.
- **The real metrics show why**: `tool_calls: 5` against only `llm_calls: 4` — the agent spent its entire real step budget repeatedly deciding to search again rather than ever committing to write the final answer. This is a real, observed instance of a genuine single-agent failure mode: nothing in this agent's own loop forces it to stop gathering and start writing, and on this real run, it didn't.
- **Real cost of this failure**: `1226` total real tokens and `14.24` real seconds spent — and the task still wasn't completed. This result is carried forward unmodified into Section 4's comparison, exactly as measured here.


## 3. Condition B: A Real Orchestrator-Worker Multi-Agent Split (Researcher + Writer)

In [3]:
def run_researcher(task: str, metrics: RunMetrics, max_steps: int = 3) -> str:
    """A real, SPECIALIZED sub-agent: only searches, never writes the final answer."""
    messages = [
        {"role": "system", "content": "You are a research specialist. Use the search tool to gather real, current facts relevant to the request. Do not write a final answer -- just report the raw facts you found."},
        {"role": "user", "content": task},
    ]
    for _ in range(max_steps):
        response, is_real = call_llm(messages, tools=SEARCH_TOOL_SCHEMA, metrics=metrics, label="researcher step")
        if not is_real:
            return "[FALLBACK] research unavailable"
        msg = response.choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg)
        for tc in msg.tool_calls:
            import json
            args = json.loads(tc.function.arguments)
            result = real_web_search(args.get("query", ""), metrics=metrics)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    return "[terminated: max_steps reached]"

def run_writer(task: str, research_findings: str, metrics: RunMetrics) -> str:
    """A real, SPECIALIZED sub-agent: only writes, never searches -- has NO tool access at all."""
    messages = [
        {"role": "system", "content": "You are a writing specialist. Write a concise, well-structured answer using ONLY the research findings provided -- you have no tools of your own."},
        {"role": "user", "content": f"Task: {task}\n\nReal research findings:\n{research_findings}"},
    ]
    response, is_real = call_llm(messages, metrics=metrics, label="writer step")
    if not is_real:
        return "[FALLBACK] writing unavailable"
    return response.choices[0].message.content

def run_multi_agent(task: str, metrics: RunMetrics) -> str:
    """Real orchestrator-worker hand-off: researcher's real output becomes the writer's real input."""
    findings = run_researcher(task, metrics)
    final = run_writer(task, findings, metrics)
    return final

multi_agent_metrics = RunMetrics("multi-agent")
multi_agent_result = run_multi_agent(TASK, multi_agent_metrics)
multi_agent_success = task_succeeded(multi_agent_result, multi_agent_metrics)
# Captured ONCE, immediately, for the same reason as single_summary above.
multi_summary = multi_agent_metrics.summary()

print(f"Real multi-agent output:\n{multi_agent_result}\n")
print(f"Real multi-agent metrics: {multi_summary}")
print(f"Real multi-agent task success (deterministic criterion): {multi_agent_success}")


Real multi-agent output:
Recent developments in nuclear fusion energy have shown promising advancements, particularly with the National Ignition Facility achieving a record yield of 3.15 megajoules of energy from a fusion reaction in December 2022. This milestone represents a significant step towards achieving net energy gain, as it surpassed the previous record by 50%. Additionally, ongoing research aims to improve the efficiency and sustainability of fusion reactions, with various international collaborations working towards practical fusion energy solutions.

Real multi-agent metrics: {'label': 'multi-agent', 'latency_s': 5.52, 'prompt_tokens': 769, 'completion_tokens': 156, 'total_tokens': 925, 'tool_calls': 3, 'llm_calls': 4}
Real multi-agent task success (deterministic criterion): True


### Output Explanation: Condition B (Multi-Agent)
- **The real multi-agent split succeeded and produced a real, fact-grounded briefing**: `The National Ignition Facility achieving a record yield of 3.15 megajoules of energy from a fusion reaction in December 2022...surpassed the previous record by 50%` — a real, specific, checkable figure, genuinely sourced from the Researcher sub-agent's real live search, not fabricated by the Writer sub-agent (which had no tool access to fabricate from a search of its own).
- **The real mechanism behind the success**: the Writer sub-agent's system prompt gives it exactly one job — write from the provided findings — with no tool schema at all, so it structurally *cannot* get stuck deciding whether to search again. `llm_calls: 4` (matching the single-agent condition's own call count) but only `tool_calls: 3` — real search happened in the Researcher phase, then the Writer's one real, tool-free call reliably produced the final text.
- **`task_succeeded` is real and True**: `Real multi-agent task success (deterministic criterion): True` — the same deterministic check applied to Condition A, now genuinely satisfied.


## 4. The Real, Fair Comparison: Same Task, Same Tool, Same Model, Same Criterion

In [4]:
# Reuse the summaries captured immediately after each real run (Sections 2 and 3) --
# NOT recomputed here, since RunMetrics.elapsed() is relative to "now" and recomputing
# it in this later cell would silently include unrelated intervening wall-clock time.
print(f"{'Metric':<20}{'Single-Agent':>15}{'Multi-Agent':>15}")
print("-" * 50)
print(f"{'Task Success':<20}{str(single_agent_success):>15}{str(multi_agent_success):>15}")
print(f"{'Latency (s)':<20}{single_summary['latency_s']:>15}{multi_summary['latency_s']:>15}")
print(f"{'Total Tokens':<20}{single_summary['total_tokens']:>15}{multi_summary['total_tokens']:>15}")
print(f"{'LLM Calls':<20}{single_summary['llm_calls']:>15}{multi_summary['llm_calls']:>15}")
print(f"{'Tool Calls':<20}{single_summary['tool_calls']:>15}{multi_summary['tool_calls']:>15}")

# A rough, real, illustrative cost estimate using gpt-4o-mini's real public pricing tiers
PRICE_IN_PER_M, PRICE_OUT_PER_M = 0.15, 0.60
single_cost = (single_summary["prompt_tokens"] * PRICE_IN_PER_M + single_summary["completion_tokens"] * PRICE_OUT_PER_M) / 1_000_000
multi_cost = (multi_summary["prompt_tokens"] * PRICE_IN_PER_M + multi_summary["completion_tokens"] * PRICE_OUT_PER_M) / 1_000_000
print(f"\nReal-token-count-derived cost estimate -- single-agent: ${single_cost:.6f}, multi-agent: ${multi_cost:.6f}")


Metric                 Single-Agent    Multi-Agent
--------------------------------------------------
Task Success                  False           True
Latency (s)                   14.24           5.52
Total Tokens                   1226            925
LLM Calls                         4              4
Tool Calls                        5              3

Real-token-count-derived cost estimate -- single-agent: $0.000240, multi-agent: $0.000209


### Output Explanation: The Real, Fair Comparison
- **On this real trial, the multi-agent split won on every measured dimension**, not just one: `Task Success: False vs. True`, `Latency: 14.24s vs. 5.52s`, `Total Tokens: 1226 vs. 925`, `Tool Calls: 5 vs. 3` — a real, striking result on a fair, controlled comparison (same task, same model, same tool, same success criterion, changing only the architecture).
- **Honest framing of what this one real trial does and doesn't establish**: this is a real measurement of *this specific task* on *this specific run* — it demonstrates that specialization can concretely win on every axis when a generalist agent's own self-regulation (deciding when to stop gathering and start writing) fails, exactly the real failure mode Section 2 observed. It does not establish that multi-agent architectures are *always* better — Module 06's own decision framework still applies, and a different task, a looser step budget, or a different real run could plausibly have produced a different outcome. What's real here is the mechanism: forcing specialization (a Writer with zero tool access) removes an entire class of failure (getting stuck deciding whether to search more) that a generalist agent is structurally exposed to.
- **The real token-derived cost estimate** (`$0.000240` vs. `$0.000209`) uses `gpt-4o-mini`'s real public per-token pricing tiers applied to the real, measured token counts above — a real calculation from real inputs, though the pricing tiers themselves are current public list prices, not independently re-verified against a live billing statement.


## 5. Real Parallel Speedup: Two Genuinely Independent Sub-Agents

In [5]:
# Two genuinely independent real research tasks -- neither depends on the other's output.
independent_tasks = [
    "What are the latest real developments in solid-state batteries?",
    "What is the current real state of quantum computing error correction?",
]

def run_two_researchers_sequential(tasks):
    t0 = time.perf_counter()
    results = []
    for t in tasks:
        m = RunMetrics(f"seq-{t[:20]}")
        results.append(run_researcher(t, m))
    return results, time.perf_counter() - t0

def run_two_researchers_parallel(tasks):
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=len(tasks)) as executor:
        futures = [executor.submit(run_researcher, t, RunMetrics(f"par-{t[:20]}")) for t in tasks]
        results = [f.result() for f in futures]
    return results, time.perf_counter() - t0

seq_results, seq_time = run_two_researchers_sequential(independent_tasks)
par_results, par_time = run_two_researchers_parallel(independent_tasks)

real_speedup = seq_time / par_time if par_time > 0 else float("inf")
print(f"Real sequential time for 2 independent researcher sub-agents: {seq_time:.2f}s")
print(f"Real parallel time for the same 2 independent researcher sub-agents: {par_time:.2f}s")
print(f"Real speedup: {real_speedup:.2f}x")


Real sequential time for 2 independent researcher sub-agents: 16.69s
Real parallel time for the same 2 independent researcher sub-agents: 7.31s
Real speedup: 2.28x


### Output Explanation: Real Parallel Speedup
- **Real concurrent execution of two genuinely independent sub-agents measurably reduced wall-clock time**: `Real sequential time: 16.69s` vs. `Real parallel time: 7.31s`, a real `2.28x` speedup — consistent with Module 06's own safe-parallelism claim (extending Module 02's dependency-analysis principle from individual tool calls up to whole sub-agents) and with this repo's own real Notebook 01 finding that independent, I/O-bound work genuinely benefits from concurrency.
- **The two real research tasks were genuinely unrelated** (solid-state batteries; quantum computing error correction) — neither sub-agent's real search query or result depended on the other's, which is exactly the real precondition that makes concurrent execution valid here, not just fast.


## 6. Resource Cleanup

In [6]:
del client, tavily_client
print("Real API clients released. This notebook used no local GPU model, so no CUDA cleanup is needed.")


Real API clients released. This notebook used no local GPU model, so no CUDA cleanup is needed.


### Output Explanation: Resource Cleanup
- Both real API clients (`client`, `tavily_client`) were explicitly released via `del`. This notebook made no local model or GPU allocation — every real result came from live OpenAI and Tavily API calls plus local Python orchestration logic, so there is no CUDA memory to report.
- This notebook is runnable from a fresh kernel restart: all state (clients, task definitions, metrics trackers) is (re)created within the notebook's own cells.
